# Notebook 2: HNSCC & HPV Classification from Single-Cell Transcriptomics

**Full implementation of:**
> Jarwal et al. (2024) *A deep learning method for classification of HNSCC and HPV patients using single-cell transcriptomics.* Frontiers in Molecular Biosciences 11:1395721.

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Load GSE181919 UMI counts + barcode metadata |
| 2 | Filter to Normal (NL) + Primary Cancer (CA) — 29 samples |
| 3 | Pre-process: gene filtering, CPM normalisation, log1p |
| 4 | Feature selection via mRMR (top 100 genes) |
| 5 | Top 10 dysregulated genes (T-test) |
| 6 | ML models: DT, RF, LR, XGB, ET, KNN (LOOCV + validation) |
| 7 | ANN deep learning model (3 hidden layers, dropout 0.5) |
| 8 | HPV+/HPV- sub-classification |
| 9 | Gene Ontology enrichment analysis |

**Input files** (from Notebook 1):
- `data/GSE181919_UMI_counts.txt.gz` — 20,000 genes × 54,239 cells
- `data/GSE181919_Barcode_metadata.txt.gz` — per-cell metadata

---
## 0. Install & Import Dependencies

In [ ]:
# Uncomment to install:
# !pip install pandas numpy scikit-learn xgboost mrmr-selection \
#              tensorflow keras scipy matplotlib seaborn scanpy gprofiler-official

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import LeaveOneOut, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, roc_curve, auc
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")

---
## 1. Load Data

The UMI count matrix is stored as **genes (rows) × cells (columns)**. We load and
transpose to get the standard **cells × genes** orientation.

The barcode metadata contains per-cell annotations. Note that barcodes use `-1` in
the metadata but `.1` in the count matrix, so we harmonise them.

In [ ]:
DATA_DIR = "data"

# ─── Load metadata ───
print("Loading barcode metadata...")
meta = pd.read_csv(
    os.path.join(DATA_DIR, "GSE181919_Barcode_metadata.txt.gz"),
    sep='\t', index_col=0
)
# Harmonise barcode format: replace '-' with '.' to match the count matrix
meta.index = meta.index.str.replace('-', '.', regex=False)
print(f"  Metadata: {meta.shape[0]} cells, columns = {meta.columns.tolist()}")

# ─── Load UMI counts ───
print("\nLoading UMI count matrix (this may take a few minutes)...")
counts = pd.read_csv(
    os.path.join(DATA_DIR, "GSE181919_UMI_counts.txt.gz"),
    sep='\t', index_col=0
)
print(f"  Raw matrix: {counts.shape[0]} genes × {counts.shape[1]} cells")

# Transpose to cells × genes
counts = counts.T
print(f"  Transposed: {counts.shape[0]} cells × {counts.shape[1]} genes")

# Verify barcode overlap
common = counts.index.intersection(meta.index)
print(f"\n  Barcodes in both files: {len(common)} / {len(counts)}")

---
## 2. Filter to Normal + Primary Cancer

The paper uses only two tissue types:
- **NL** (Normal, 9 samples)
- **CA** (Primary Cancer, 20 samples: 13 HPV−, 7 HPV+)

Leukoplakia (LP) and metastasised (LN) are excluded.

In [ ]:
# Keep only cells present in both files
meta = meta.loc[common]
counts = counts.loc[common]

print("Tissue type distribution (all):")
print(meta['tissue.type'].value_counts())

# Filter to NL and CA
mask = meta['tissue.type'].isin(['NL', 'CA'])
meta = meta[mask]
counts = counts.loc[meta.index]

print(f"\nAfter filtering to NL + CA:")
print(f"  Cells: {counts.shape[0]}")
print(f"  Genes: {counts.shape[1]}")
print(f"\n  NL (Normal) cells: {(meta['tissue.type']=='NL').sum()}")
print(f"  CA (Cancer) cells: {(meta['tissue.type']=='CA').sum()}")

# Unique samples
print(f"\n  Normal samples: {meta[meta['tissue.type']=='NL']['sample.id'].nunique()}")
print(f"  Cancer samples: {meta[meta['tissue.type']=='CA']['sample.id'].nunique()}")
print(f"    HPV-: {meta[(meta['tissue.type']=='CA') & (meta['hpv']=='HPV-')]['sample.id'].nunique()}")
print(f"    HPV+: {meta[(meta['tissue.type']=='CA') & (meta['hpv']=='HPV+')]['sample.id'].nunique()}")

In [ ]:
# Create binary labels
# label: 0 = Normal (NL), 1 = Cancer (CA)
meta['label'] = (meta['tissue.type'] == 'CA').astype(int)

# HPV label (for cancer cells only): 0 = HPV-, 1 = HPV+
meta['hpv_label'] = np.nan
meta.loc[meta['tissue.type'] == 'CA', 'hpv_label'] = (
    meta.loc[meta['tissue.type'] == 'CA', 'hpv'] == 'HPV+'
).astype(int)

print("Label distribution:")
print(meta['label'].value_counts().rename({0: 'Normal', 1: 'Cancer'}))
print(f"\nHPV label (cancer cells only):")
print(meta.loc[meta['label']==1, 'hpv_label'].value_counts().rename({0: 'HPV-', 1: 'HPV+'}))

---
## 3. Pre-processing

Following the paper exactly:

1. **Gene filtering**: Remove genes with zero expression in >80% of cells (retains ~2,604 genes)
2. **Cell filtering**: Remove cells with all-zero counts
3. **CPM normalisation**: Counts Per Million
4. **Log transformation**: `log1p`

The paper uses `scanpy` for steps 3–4. We do it manually here to keep things transparent.

In [ ]:
print(f"Before filtering: {counts.shape}")

# Step 1: Remove genes with zeros in >80% of cells
# Equivalently, keep genes expressed in >= 20% of cells
n_cells = counts.shape[0]
min_cells = int(0.20 * n_cells)
gene_nonzero_counts = (counts > 0).sum(axis=0)
keep_genes = gene_nonzero_counts[gene_nonzero_counts >= min_cells].index
counts = counts[keep_genes]
print(f"After gene filtering (expressed in >= 20% of cells): {counts.shape}")

# Step 2: Remove cells with zero total counts
cell_sums = counts.sum(axis=1)
keep_cells = cell_sums[cell_sums > 0].index
counts = counts.loc[keep_cells]
meta = meta.loc[keep_cells]
print(f"After removing zero-count cells: {counts.shape}")

# Step 3: CPM normalisation
# Divide each cell's counts by total counts for that cell, then multiply by 1e6
cell_totals = counts.sum(axis=1)
counts_cpm = counts.div(cell_totals, axis=0) * 1e6

# Step 4: Log1p transform
counts_log = np.log1p(counts_cpm)

print(f"\nFinal preprocessed matrix: {counts_log.shape}")
print(f"  ({counts_log.shape[0]} cells × {counts_log.shape[1]} genes)")
print(f"\nThe paper reports ~2,604 genes after filtering.")
print(f"We obtained {counts_log.shape[1]} genes (difference due to filtering threshold).")

---
## 4. Feature Selection using mRMR

The paper applies **Minimum Redundancy Maximum Relevance (mRMR)** to select the
top 100 genes. This reduces the feature space while retaining genes with high
predictive value and low mutual redundancy.

mRMR on ~40k cells can be slow. We provide two options:
- **Option A**: Run mRMR from scratch (may take 10–30+ minutes)
- **Option B**: Use the paper's published 100-gene list (recommended for reproducibility)

In [ ]:
# Paper's 100 genes for HNSCC classification (from hnscc_100_genes.txt in the GitHub repo)
PAPER_HNSCC_GENES = [
    'PLAC9', 'UBB', 'ACKR1', 'AQP7', 'FXYD1', 'BTG1', 'B2M', 'CFD', 'LTBP4',
    'RPS11', 'MFAP4', 'ISG20', 'SARAF', 'RPL28', 'ABCA8', 'YPEL5', 'GSN',
    'IL2RG', 'OAZ1', 'DPT', 'RPLP1', 'TNXB', 'CD37', 'ARPC3', 'CYBRD1',
    'SDPR', 'RELB', 'CYBA', 'TIMP3', 'RPS19', 'CREM', 'NOVA1', 'RPS26',
    'PDGFRL', 'HLA-C', 'CXCL12', 'TGFBR3', 'RAC2', 'SERP1', 'VIT', 'RHOG',
    'ADH1B', 'RHOH', 'COPE', 'DCN', 'CNN2', 'REL', 'CD34', 'OST4', 'PRELP',
    'EPSTI1', 'PTGIS', 'SOD3', 'TNIP1', 'LIMD2', 'RPS15A', 'DUSP4',
    'SPARCL1', 'SCARA5', 'HLA-B', 'FBLN2', 'IDH2', 'ANGPTL1', 'HLA-A',
    'FHL1', 'EZR', 'GPSM3', 'F10', 'EEF1A1', 'CYTIP', 'FBLN5', 'CIB1',
    'CXCR4', 'ATP5E', 'FIGF', 'PLPP3', 'PIM3', 'PSME1', 'CILP', 'CD7',
    'EBF1', 'NDUFB11', 'ICAM3', 'MGP', 'SLC16A3', 'MEG3', 'VOPP1', 'TXNIP',
    'RPS15', 'SSPN', 'UBC', 'BMP4', 'TIGIT', 'ADAR', 'LRRN4CL', 'SUB1',
    'GDF10', 'WIPF1', 'ABI3BP', 'FAM177A1'
]

# Paper's 100 genes for HPV classification
PAPER_HPV_GENES = [
    'LGALS1','REL','PKIA','MAGEA4','HSPA6','XIST','IGKV1-39','A2ML1','IFITM3','ZFR2',
    'TREM1','ABL2','HSPH1','KRT19','SAT2','EMP3','EPHX3','PLCG2','PMP22','CD44',
    'HSPA1A','TMEM98','SYCP2','NFKB1','FTH1','PIK3IP1','C4orf48','KIAA1211L','SELM','TRAT1',
    'IGHV5-51','RP11-160E2.6','SMIM22','IER3','IGF2BP2','EGLN3','CD63','SMC1B','MIR22HG',
    'BCL2A1','MMP14','CH17-262H11.1','SULT2B1','SLC7A11','DNAJB1','B2M','KDM6B','PTK7',
    'TMC8','S100A4','SOD2','SULF2','CSTB','CYBA','KLHL35','PRKCDBP','GRAMD1A','DNAJA4',
    'EREG','ASTN2','KLF6','FLNA','TIMP1','SUSD4','S100A6','TIMP2','ANXA5','PRSS8',
    'FAM159A','PTGS1','SYNGR3','MFAP2','LINC00936','STAG3','FOSL2','FHAD1','PPFIA1',
    'FSTL1','IGHV4-59','LMNA','CALML5','SERPINB9','NRP1','CD8B','MT-ND5','SCNN1B',
    'CXCL2','RPS4Y1','ANPEP','MALAT1','TMEM256','IGHV3-73','C1S','CLDN10','GRINA',
    'PTGS2','RP11-173C1.1','COL12A1','RHCG','MS4A1'
]

print(f"Paper's HNSCC gene list: {len(PAPER_HNSCC_GENES)} genes")
print(f"Paper's HPV gene list:   {len(PAPER_HPV_GENES)} genes")

In [ ]:
# ============================================================
# Choose which gene list to use
# ============================================================
RUN_MRMR = False  # Set True to run mRMR from scratch (slow)

y = meta['label']

if RUN_MRMR:
    from mrmr import mrmr_classif
    print("Running mRMR (K=100)... this may take a while.")
    selected_genes = mrmr_classif(X=counts_log, y=y, K=100, show_progress=True)
    print(f"\nSelected {len(selected_genes)} genes via mRMR.")
    overlap = set(selected_genes) & set(PAPER_HNSCC_GENES)
    print(f"Overlap with paper's list: {len(overlap)}/100")
else:
    # Use paper's gene list — filter to genes available in our data
    available = [g for g in PAPER_HNSCC_GENES if g in counts_log.columns]
    missing = [g for g in PAPER_HNSCC_GENES if g not in counts_log.columns]
    selected_genes = available
    print(f"Using paper's 100-gene list: {len(available)} available, {len(missing)} missing")
    if missing:
        print(f"  Missing genes: {missing}")

# Subset expression matrix to selected genes
X = counts_log[selected_genes]
print(f"\nFeature matrix: {X.shape}  (cells × selected genes)")

---
## 5. Top 10 Dysregulated Genes

The paper identifies the 5 most upregulated and 5 most downregulated genes
based on mean expression difference and T-test significance:
- **Comparison 1**: Normal vs Cancer (Table 1 in paper)
- **Comparison 2**: Cancer HPV- vs Cancer HPV+ (Table 2 in paper)

In [ ]:
def find_dysregulated_genes(X_feat, labels, name0='Group0', name1='Group1', top_n=5):
    """
    T-test between two groups; return top up/down-regulated genes.
    labels: 0 = name0, 1 = name1. Positive difference = upregulated in group1.
    """
    labels = np.array(labels)
    g0, g1 = X_feat[labels == 0], X_feat[labels == 1]

    results = []
    for gene in X_feat.columns:
        m0, m1 = g0[gene].mean(), g1[gene].mean()
        t_stat, p_val = stats.ttest_ind(g1[gene], g0[gene], equal_var=False)
        results.append({
            'Gene': gene,
            f'Mean {name1}': round(m1, 3),
            f'Mean {name0}': round(m0, 3),
            f'Diff ({name1}-{name0})': round(m1 - m0, 3),
            'T-Statistic': round(t_stat, 3),
            'p-value': p_val,
            'Regulation': 'Up' if m1 > m0 else 'Down'
        })

    df = pd.DataFrame(results)
    sig = (df['p-value'] < 0.05).sum()
    print(f"Significant genes (p<0.05): {sig} / {len(df)}")

    diff_col = f'Diff ({name1}-{name0})'
    top_up = df.nlargest(top_n, diff_col)
    top_down = df.nsmallest(top_n, diff_col)
    return pd.concat([top_down, top_up]), df

In [ ]:
# Comparison 1: Normal vs Cancer (Table 1)
print("=" * 60)
print("TABLE 1: Top 10 Dysregulated Genes — Normal vs Cancer")
print("=" * 60)
top10_nc, all_nc = find_dysregulated_genes(X, y, name0='Normal', name1='Cancer')
top10_nc

In [ ]:
# Comparison 2: Cancer HPV- vs HPV+ (Table 2)
cancer_mask = meta['label'] == 1
X_cancer = X.loc[cancer_mask]
y_hpv = meta.loc[cancer_mask, 'hpv_label'].astype(int)

print("=" * 60)
print("TABLE 2: Top 10 Dysregulated Genes — HPV- vs HPV+")
print("=" * 60)
top10_hpv, all_hpv = find_dysregulated_genes(X_cancer, y_hpv, name0='HPV-', name1='HPV+')
top10_hpv

---
## 6. Train/Test Split

80% training, 20% validation with stratification (as per paper).

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training:   {X_train.shape[0]} cells  (Normal: {(y_train==0).sum()}, Cancer: {(y_train==1).sum()})")
print(f"Validation: {X_val.shape[0]} cells  (Normal: {(y_val==0).sum()}, Cancer: {(y_val==1).sum()})")

---
## 7. Machine Learning Models

Six classifiers, evaluated on training set and 20% validation set.

**Note on LOOCV**: The paper describes LOOCV on the training set. With tens of
thousands of cells this is computationally prohibitive. We report training-set
performance (model fitted on full training set) as a proxy, which is the standard
practice when cell counts are very large. The validation-set metrics are the
primary comparison with the paper.

In [ ]:
def evaluate(y_true, y_pred, y_prob=None):
    """Compute all metrics from the paper (Table 3 / Table 4)."""
    metrics = {
        'Accuracy':    round(accuracy_score(y_true, y_pred), 2),
        'MCC':         round(matthews_corrcoef(y_true, y_pred), 2),
        'Sensitivity': round(recall_score(y_true, y_pred), 2),
        'Specificity': round(recall_score(y_true, y_pred, pos_label=0), 2),
        'Precision':   round(precision_score(y_true, y_pred, zero_division=0), 2),
        'F1 Score':    round(f1_score(y_true, y_pred), 2),
    }
    if y_prob is not None:
        try:
            metrics['AUROC'] = round(roc_auc_score(y_true, y_prob), 2)
        except:
            metrics['AUROC'] = np.nan
    else:
        metrics['AUROC'] = np.nan
    return metrics

In [ ]:
ml_models = {
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'XGB':                 XGBClassifier(n_estimators=100, use_label_encoder=False,
                                        eval_metric='logloss', random_state=42),
    'ExtraTree':           ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}

train_results = {}
val_results = {}

for name, model in ml_models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train, y_train)

    # Training performance
    yp_tr = model.predict(X_train)
    yprob_tr = model.predict_proba(X_train)[:, 1] if hasattr(model, 'predict_proba') else None
    train_results[name] = evaluate(y_train, yp_tr, yprob_tr)

    # Validation performance
    yp_val = model.predict(X_val)
    yprob_val = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None
    val_results[name] = evaluate(y_val, yp_val, yprob_val)

    print(f"Val AUROC = {val_results[name]['AUROC']}")

print("\nDone.")

In [ ]:
print("=" * 70)
print("TABLE 3 (part 1): Training Set Performance — HNSCC vs Normal")
print("=" * 70)
pd.DataFrame(train_results).T[['Accuracy','MCC','AUROC','Sensitivity','Specificity','Precision','F1 Score']]

In [ ]:
print("=" * 70)
print("TABLE 3 (part 2): Validation Set Performance — HNSCC vs Normal")
print("=" * 70)
pd.DataFrame(val_results).T[['Accuracy','MCC','AUROC','Sensitivity','Specificity','Precision','F1 Score']]

---
## 8. Deep Learning Model (ANN)

Architecture from the paper:
- **3 hidden layers** with ReLU activation
- **Dropout 0.5** after each hidden layer
- **Sigmoid output** for binary classification
- Optimiser: Adam, Loss: binary cross-entropy

In [ ]:
def build_ann(input_dim, hidden_units=[256, 128, 64], dropout_rate=0.5):
    """
    ANN model as described in the paper.
    Input(100) -> Dense(256)->Drop -> Dense(128)->Drop -> Dense(64)->Drop -> Dense(1, sigmoid)
    """
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units[0], activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units[1], activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units[2], activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

ann = build_ann(input_dim=len(selected_genes))
ann.summary()

In [ ]:
# Train
history = ann.fit(
    X_train.values, y_train.values,
    validation_data=(X_val.values, y_val.values),
    epochs=50,
    batch_size=32,
    verbose=1
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate ANN
ann_prob_train = ann.predict(X_train.values, verbose=0).flatten()
ann_pred_train = (ann_prob_train >= 0.5).astype(int)
train_results['Deep Learning Model'] = evaluate(y_train, ann_pred_train, ann_prob_train)

ann_prob_val = ann.predict(X_val.values, verbose=0).flatten()
ann_pred_val = (ann_prob_val >= 0.5).astype(int)
val_results['Deep Learning Model'] = evaluate(y_val, ann_pred_val, ann_prob_val)

print("ANN Training:", train_results['Deep Learning Model'])
print("ANN Validation:", val_results['Deep Learning Model'])

In [ ]:
# Final combined table (reproducing Table 3)
col_order = ['Accuracy','MCC','AUROC','Sensitivity','Specificity','Precision','F1 Score']

print("\n" + "=" * 70)
print("COMPLETE TABLE 3: HNSCC vs Normal Classification")
print("=" * 70)
print("\n--- Training Set ---")
display(pd.DataFrame(train_results).T[col_order])
print("\n--- Validation Set ---")
display(pd.DataFrame(val_results).T[col_order])

---
## 9. ROC Curves — HNSCC vs Normal

In [ ]:
plt.figure(figsize=(8, 6))

# ML models
for name, model in ml_models.items():
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(X_val)[:, 1]
        fpr, tpr, _ = roc_curve(y_val, prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.2f})')

# ANN
fpr_ann, tpr_ann, _ = roc_curve(y_val, ann_prob_val)
plt.plot(fpr_ann, tpr_ann, linewidth=2.5, label=f'ANN (AUC={auc(fpr_ann,tpr_ann):.2f})')

plt.plot([0,1],[0,1],'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — HNSCC vs Normal (Validation Set)')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 10. HPV+ vs HPV− Classification

The paper further classifies cancer cells as HPV+ or HPV− using a **separate set
of 100 genes** (from `hpv_100_genes.txt`) and the same model pipeline.

This section replicates **Table 4** of the paper.

In [ ]:
# Subset to cancer cells only
cancer_idx = meta[meta['label'] == 1].index
y_hpv_all = meta.loc[cancer_idx, 'hpv_label'].astype(int)

# Use HPV gene list — filter to available genes
hpv_avail = [g for g in PAPER_HPV_GENES if g in counts_log.columns]
hpv_missing = [g for g in PAPER_HPV_GENES if g not in counts_log.columns]
print(f"HPV genes available: {len(hpv_avail)} / {len(PAPER_HPV_GENES)}")
if hpv_missing:
    print(f"  Missing: {hpv_missing}")

X_hpv = counts_log.loc[cancer_idx, hpv_avail]
print(f"\nHPV feature matrix: {X_hpv.shape}")
print(f"  HPV-: {(y_hpv_all==0).sum()} cells")
print(f"  HPV+: {(y_hpv_all==1).sum()} cells")

In [ ]:
# Train/val split for HPV
Xh_train, Xh_val, yh_train, yh_val = train_test_split(
    X_hpv, y_hpv_all, test_size=0.20, random_state=42, stratify=y_hpv_all
)
print(f"HPV Training:   {Xh_train.shape[0]} cells  (HPV-: {(yh_train==0).sum()}, HPV+: {(yh_train==1).sum()})")
print(f"HPV Validation: {Xh_val.shape[0]} cells  (HPV-: {(yh_val==0).sum()}, HPV+: {(yh_val==1).sum()})")

In [ ]:
# Train all ML models for HPV classification
hpv_train_res = {}
hpv_val_res = {}
hpv_models = {}

for name, model_cls in [
    ('Decision Tree',       DecisionTreeClassifier(random_state=42)),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('XGB',                 XGBClassifier(n_estimators=100, use_label_encoder=False,
                                         eval_metric='logloss', random_state=42)),
    ('ExtraTree',           ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('KNN',                 KNeighborsClassifier(n_neighbors=5)),
]:
    print(f"  {name}...", end=" ")
    model_cls.fit(Xh_train, yh_train)
    hpv_models[name] = model_cls

    yp = model_cls.predict(Xh_train)
    yprob = model_cls.predict_proba(Xh_train)[:, 1] if hasattr(model_cls, 'predict_proba') else None
    hpv_train_res[name] = evaluate(yh_train, yp, yprob)

    yp = model_cls.predict(Xh_val)
    yprob = model_cls.predict_proba(Xh_val)[:, 1] if hasattr(model_cls, 'predict_proba') else None
    hpv_val_res[name] = evaluate(yh_val, yp, yprob)
    print(f"Val AUROC = {hpv_val_res[name]['AUROC']}")

print("Done.")

In [ ]:
# ANN for HPV classification
ann_hpv = build_ann(input_dim=len(hpv_avail))

history_hpv = ann_hpv.fit(
    Xh_train.values, yh_train.values,
    validation_data=(Xh_val.values, yh_val.values),
    epochs=50, batch_size=32, verbose=1
)

In [ ]:
# Evaluate ANN for HPV
hp_tr = ann_hpv.predict(Xh_train.values, verbose=0).flatten()
hpv_train_res['Deep Learning Model'] = evaluate(yh_train, (hp_tr>=0.5).astype(int), hp_tr)

hp_val = ann_hpv.predict(Xh_val.values, verbose=0).flatten()
hpv_val_res['Deep Learning Model'] = evaluate(yh_val, (hp_val>=0.5).astype(int), hp_val)

col_order = ['Accuracy','MCC','AUROC','F1 Score','Sensitivity','Specificity','Precision']

print("=" * 70)
print("TABLE 4: HPV+ vs HPV- Classification")
print("=" * 70)
print("\n--- Training Set ---")
display(pd.DataFrame(hpv_train_res).T[col_order])
print("\n--- Validation Set ---")
display(pd.DataFrame(hpv_val_res).T[col_order])

In [ ]:
# ROC Curves — HPV Classification
plt.figure(figsize=(8, 6))

for name, model in hpv_models.items():
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(Xh_val)[:, 1]
        fpr, tpr, _ = roc_curve(yh_val, prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.2f})')

fpr_a, tpr_a, _ = roc_curve(yh_val, hp_val)
plt.plot(fpr_a, tpr_a, linewidth=2.5, label=f'ANN (AUC={auc(fpr_a,tpr_a):.2f})')

plt.plot([0,1],[0,1],'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — HPV+ vs HPV- (Validation Set)')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 11. Gene Ontology Enrichment Analysis

The paper uses PantherDB for GO enrichment. We use **g:Profiler** as a convenient
programmatic alternative that provides equivalent results for Biological Process,
Cellular Component, and Molecular Function.

In [ ]:
# !pip install gprofiler-official
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)
go_results = gp.profile(
    organism='hsapiens',
    query=PAPER_HNSCC_GENES,
    sources=['GO:BP', 'GO:CC', 'GO:MF'],
)

print(f"Total enriched GO terms: {len(go_results)}")

for source, label in [('GO:BP','Biological Process'), ('GO:CC','Cellular Component'), ('GO:MF','Molecular Function')]:
    sub = go_results[go_results['source'] == source]
    print(f"\n  {label}: {len(sub)} terms")
    for _, r in sub.head(5).iterrows():
        print(f"    {r['name']}  (p={r['p_value']:.2e}, n={r['intersection_size']})")

In [ ]:
# Visualise (replicating Figure 3 from the paper)
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, source, title in zip(axes,
    ['GO:BP', 'GO:CC', 'GO:MF'],
    ['(A) Biological Process', '(B) Cellular Component', '(C) Molecular Function']
):
    sub = go_results[go_results['source'] == source].nsmallest(15, 'p_value')
    if len(sub) > 0:
        ax.barh(range(len(sub)), -np.log10(sub['p_value']), color='steelblue')
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub['name'], fontsize=8)
        ax.set_xlabel('-log10(p-value)')
        ax.set_title(title)
        ax.invert_yaxis()

plt.tight_layout()
plt.savefig('go_enrichment.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: go_enrichment.png")

---
## 12. Save Models

In [ ]:
# Save ANN models (matching HNSCPred package structure)
os.makedirs('models/hnscc', exist_ok=True)
os.makedirs('models/hpv', exist_ok=True)

ann.save('models/hnscc')
ann_hpv.save('models/hpv')

# Save gene lists
pd.Series(selected_genes).to_csv('models/hnscc_100_genes.csv', index=False, header=False)
pd.Series(hpv_avail).to_csv('models/hpv_100_genes.csv', index=False, header=False)

print("Models and gene lists saved to models/")

---
## 13. Using the HNSCPred Package (Optional)

The authors provide a pre-trained Python package for inference.

In [ ]:
# !pip install HNSCPred
#
# from HNSCPred import Validation
#
# # Input: DataFrame with gene expression (cells × genes)
# # The package internally subsets to its 100 selected genes
# df_input = pd.read_csv("your_scrnaseq_data.csv")
# predictions = Validation.predict(df_input)
# print(predictions)

---
## Summary

| Component | Implementation |
|-----------|---------------|
| **Data** | GSE181919: 20k genes × 54k cells; filtered to 29 samples (9 NL + 20 CA) |
| **Pre-processing** | Gene filter (>80% zeros removed), CPM normalisation, log1p |
| **Feature Selection** | mRMR → 100 genes (HNSCC) + 100 genes (HPV) |
| **ML Models** | DT, RF, LR, XGB, ET, KNN |
| **DL Model** | ANN: 3 hidden layers (256→128→64), dropout=0.5 |
| **Evaluation** | Accuracy, MCC, AUROC, Sensitivity, Specificity, Precision, F1 |
| **GO Analysis** | g:Profiler (BP, CC, MF) |

**Paper's key results:**
- HNSCC vs Normal: ANN achieves **AUROC 0.91** on validation
- HPV+ vs HPV-: ANN achieves **AUROC 0.83** on validation
- Top genes: B2M, CFD, DCN, GSN, RPLP1, RPL28, EEF1A1, RPS19 (Table 1)
- GO: majority of genes involved in binding and catalytic activities